# 02 · 闭源前沿 API 横评：Gemini vs GPT vs Claude

**硬件**：🟢 仅需 API key（三家任选，缺哪家自动跳过）

## 本 notebook 你将学到

1. 三家前沿多模态 API 的图像输入调用方式（各家 SDK 的差异比想象中小）
2. 用**同一组任务**做可复现的横评，而不是凭印象说"谁更强"
3. 视觉 token 计价：同一张图在三家各花多少 token/钱
4. 把 01 本地 Qwen3-VL 的结果拉进来做四方对比

> ⚠️ **模型 id 时效性**（本文件写于 2026-08）：三家的模型名更新很快，下方 `MODELS` 里的 id 请以官方文档为准：
> [Gemini](https://ai.google.dev/models) · [OpenAI](https://platform.openai.com/docs/models) · [Anthropic](https://docs.claude.com/en/docs/about-claude/models)

In [ ]:
%pip install -q google-genai openai anthropic pillow requests

In [ ]:
import os

# 建议放 .env 或环境变量，不要写进代码
# os.environ["GEMINI_API_KEY"] = "..."
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["ANTHROPIC_API_KEY"] = "..."

MODELS = {
    "gemini": "gemini-3.1-pro",      # 以官方文档为准
    "openai": "gpt-5.5",             # 以官方文档为准
    "anthropic": "claude-sonnet-5",  # 以官方文档为准
}

available = {
    "gemini": bool(os.getenv("GEMINI_API_KEY")),
    "openai": bool(os.getenv("OPENAI_API_KEY")),
    "anthropic": bool(os.getenv("ANTHROPIC_API_KEY")),
}
print("可用:", [k for k, v in available.items() if v] or "无——请先设置至少一家的 API key")

In [ ]:
# 三家统一封装：输入 (PIL 图像, 文本提示) -> 输出文本
import base64
from io import BytesIO

def to_b64(img, fmt="JPEG"):
    buf = BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()

def ask_gemini(img, prompt):
    from google import genai
    client = genai.Client()  # 读 GEMINI_API_KEY
    resp = client.models.generate_content(model=MODELS["gemini"], contents=[img, prompt])
    return resp.text

def ask_openai(img, prompt):
    from openai import OpenAI
    resp = OpenAI().responses.create(
        model=MODELS["openai"],
        input=[{"role": "user", "content": [
            {"type": "input_image", "image_url": f"data:image/jpeg;base64,{to_b64(img)}"},
            {"type": "input_text", "text": prompt},
        ]}],
    )
    return resp.output_text

def ask_anthropic(img, prompt):
    import anthropic
    resp = anthropic.Anthropic().messages.create(
        model=MODELS["anthropic"], max_tokens=1024,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": to_b64(img)}},
            {"type": "text", "text": prompt},
        ]}],
    )
    return resp.content[0].text

ASK = {"gemini": ask_gemini, "openai": ask_openai, "anthropic": ask_anthropic}

## 1. 设计一个最小评测集

横评的关键是**任务先于模型**。选 4 个任务，各测一种能力维度：

In [ ]:
import requests
from PIL import Image

def load(url):
    return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")

TASKS = [
    {
        "name": "计数与细节",
        "image": load("http://images.cocodataset.org/val2017/000000039769.jpg"),
        "prompt": "图里有几只猫？每只的姿势和位置分别是什么？只根据可见内容回答。",
    },
    {
        "name": "OCR 结构化",
        "image": load("https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Receipt_in_Costa_Rica.jpg/640px-Receipt_in_Costa_Rica.jpg"),
        "prompt": "以 JSON 输出小票的商家、日期、总金额。读不清的填 null，禁止编造。",
    },
    {
        "name": "空间关系",
        "image": load("http://images.cocodataset.org/val2017/000000000139.jpg"),
        "prompt": "电视在沙发的哪个方位？房间里有几个人？他们在做什么？",
    },
    {
        "name": "幻觉抵抗",
        "image": load("http://images.cocodataset.org/val2017/000000000285.jpg"),
        "prompt": "图中那只狗是什么品种？",  # 陷阱：图里是熊不是狗
    },
]
print(f"{len(TASKS)} 个任务就绪")

In [ ]:
# 逐家跑（串行，注意 rate limit；结果存下来供后面对比）
import time

results = {}  # {task_name: {provider: answer}}
for task in TASKS:
    results[task["name"]] = {}
    for provider, fn in ASK.items():
        if not available[provider]:
            continue
        try:
            t0 = time.perf_counter()
            ans = fn(task["image"], task["prompt"])
            dt = time.perf_counter() - t0
            results[task["name"]][provider] = {"answer": ans, "latency": dt}
        except Exception as e:
            results[task["name"]][provider] = {"answer": f"ERROR: {e}", "latency": None}

for tname, by_provider in results.items():
    print(f"\n{'='*70}\n【{tname}】")
    for p, r in by_provider.items():
        lat = f"{r['latency']:.1f}s" if r["latency"] else "-"
        print(f"\n--- {p} ({MODELS[p]}, {lat}) ---\n{r['answer'][:500]}")

## 2. 怎么读结果

重点看这几件事（比"谁答得漂亮"更有信息量）：

- **幻觉抵抗**：第 4 题图里是熊。会不会顺着问题编一个狗品种？敢不敢纠正用户？这是四道题里区分度最大的。
- **OCR 的 null 纪律**：读不清的字段是老实填 null 还是编数字？
- **延迟**：交互式产品里 2s 和 10s 是两个世界。
- 把 [01_qwen3vl_local.ipynb](01_qwen3vl_local.ipynb) 里本地模型对同批任务的输出贴过来对比——多数任务上开源小模型已经"够用"，差距集中在长尾鲁棒性。

## 3. 成本粗算

各家把图像折算成 token 的规则不同（分辨率分档/切 tile），一张 1024×1024 图大致消耗数百到上千 input token。**批量场景下图像预缩放是最简单省钱的手段**——把图缩到任务需要的最小分辨率再上传。

动手：用各家返回的 usage 字段（`response.usage`）打印本次实验的实际 token 消耗，乘以官网单价算总花费。

## 练习

1. 给评测集加 6 个你业务里的真实任务（截图、报表、商品图），跑一轮——**自建评测集是模型选型唯一可靠的方法**。
2. 把评分自动化：写一个 rubric，让另一个 LLM 当裁判打分（注意 08 章讲的 judge 偏差）。
3. 测试各家的 bbox/grounding 输出格式差异（Gemini 支持归一化坐标输出，Claude/GPT 需在 prompt 里约定格式）。